# LM-Polygraph White-Box UQ with MedGemma

This notebook runs MedGemma clinical prompts through official LM-Polygraph white-box estimators.

It keeps raw LM-Polygraph scores, displays sampled generations, and adds a separate MinMax/Quantile confidence view using LM-Polygraph normalizers. No uncertainty formulas are manually reimplemented.

## 1. Install and authenticate

Run this once in Colab. MedGemma access requires a Hugging Face token with access to `google/medgemma-4b-it`.

In [1]:
%pip install -q "git+https://github.com/IINemo/lm-polygraph.git" transformers accelerate scipy matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.9/256.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.5 MB/s 

In [2]:
from huggingface_hub import login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN found. If the model is gated, run: login(token='...')")

Logged in to Hugging Face Hub.


## 2. Configuration and model loading

`attn_implementation="eager"` is kept because Attention Score needs attention tensors.

In [3]:
import os
import time
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from lm_polygraph.utils.model import WhiteboxModel
from lm_polygraph.utils.generation_parameters import GenerationParameters

warnings.filterwarnings("ignore")

MODEL_NAME = "google/medgemma-4b-it"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_NEW_TOKENS = 96
TEMPERATURE = 0.7
TOP_P = 0.9
SEED = 42

OUTPUT_DIR = Path("lmpolygraph_medgemma_clinical_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

Device: cuda
Model: google/medgemma-4b-it


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    attn_implementation="eager",
)
base_model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

lm_polygraph_model = WhiteboxModel(
    base_model,
    tokenizer,
    model_path=MODEL_NAME,
    instruct=True,
)

text_config = getattr(base_model.config, "text_config", base_model.config)
ATTENTION_LAYER = getattr(text_config, "num_hidden_layers", 0) // 2

print("MedGemma loaded and wrapped with LM-Polygraph WhiteboxModel.")
print("AttentionScore layer:", ATTENTION_LAYER)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

MedGemma loaded and wrapped with LM-Polygraph WhiteboxModel.
AttentionScore layer: 17


## 3. Clinical prompts and references

In [5]:
_PREFIX = "Answer concisely in 2-3 sentences covering only the key clinical points.\n\n"

PROMPTS = {
    "p01": _PREFIX + (
        "A 55-year-old man presents with sudden-onset crushing substernal chest pain "
        "radiating to the left arm, diaphoresis, and nausea. ECG shows ST-segment "
        "elevation in leads V1-V4 with reciprocal depression in II, III, aVF. "
        "Troponin I is 4.2 ng/mL (reference <0.04). What is the diagnosis and the "
        "immediate reperfusion management priority?"
    ),
    "p02": _PREFIX + (
        "A 7-year-old boy presents with a petechial rash, fever of 39.8C, neck "
        "stiffness, and photophobia. CSF shows neutrophilic pleocytosis, low glucose, "
        "and elevated protein; Gram stain reveals Gram-negative diplococci. Which "
        "organism is responsible and what is the first-line empirical antibiotic?"
    ),
    "p03": _PREFIX + (
        "Explain the mechanism by which metformin lowers blood glucose in type 2 "
        "diabetes, specifically its effect on hepatic gluconeogenesis via AMP-activated "
        "protein kinase (AMPK) and inhibition of mitochondrial complex I."
    ),
    "p04": _PREFIX + (
        "A 68-year-old woman with a 40-pack-year smoking history presents with "
        "progressive dyspnoea, 6 kg weight loss over 3 months, hoarseness, and a "
        "right-sided pleural effusion. CT reveals a 3.2 cm spiculated right upper lobe "
        "mass with ipsilateral mediastinal lymphadenopathy. What is the most likely "
        "diagnosis, and which single investigation best establishes nodal staging to "
        "guide resectability?"
    ),
    "p05": _PREFIX + (
        "A 34-year-old woman has a 2-year history of episodic bloody diarrhoea and "
        "abdominal cramping. Colonoscopy shows continuous mucosal inflammation from the "
        "rectum to the splenic flexure with pseudopolyps; biopsy confirms crypt "
        "abscesses. She has failed mesalazine. What is the next therapeutic step and "
        "the rationale for escalation?"
    ),
    "p06": _PREFIX + (
        "A 72-year-old man with CKD (eGFR 28 mL/min/1.73m2) and heart failure (EF 35%) "
        "is started on an ACE inhibitor. Two weeks later his potassium is 6.1 mEq/L and "
        "creatinine has risen 35%. Explain the pathophysiological mechanism and outline "
        "how you would manage this."
    ),
    "p07": _PREFIX + (
        "Describe the genetic basis, key clinical phenotype, and anaesthetic "
        "implications of malignant hyperthermia. Which triggering agents must be "
        "avoided, and what is the mechanism of action of dantrolene in acute management?"
    ),
    "p08": _PREFIX + (
        "A 28-year-old woman presents with recurrent pregnancy loss, livedo reticularis, "
        "and a DVT. Lupus anticoagulant and anti-cardiolipin IgG are positive on two "
        "occasions 12 weeks apart. Discuss the immunopathological mechanism of "
        "thrombosis in antiphospholipid syndrome and compare the evidence for warfarin "
        "versus DOACs for long-term anticoagulation in this population."
    ),
    "p09": _PREFIX + (
        "Compare the molecular mechanisms of acquired resistance to the third-generation "
        "EGFR tyrosine kinase inhibitor osimertinib in NSCLC, focusing on the C797S "
        "mutation, MET amplification, and small-cell transformation. For each, state the "
        "therapeutic strategy with the strongest current clinical evidence."
    ),
    "p10": _PREFIX + (
        "Describe the exact molecular mechanism by which the investigational compound "
        "XR-7291 selectively disrupts cardiolipin remodelling in the inner mitochondrial "
        "membrane of drug-resistant glioblastoma stem cells, producing selective "
        "apoptosis without harming normal neural progenitors. Name the key downstream "
        "effectors and the proposed biomarker of response."
    ),
    "p11": _PREFIX + (
        "A patient asks whether amoxicillin is an appropriate treatment for an uncomplicated "
        "viral upper respiratory tract infection because antibiotics kill viruses. Explain "
        "whether this claim is true or false and what the appropriate management should be."
    ),
}

PROMPT_IDS = list(PROMPTS.keys())
PROMPT_LIST = list(PROMPTS.values())
print(f"OK: {len(PROMPTS)} prompts loaded -> {PROMPT_IDS}")

OK: 11 prompts loaded -> ['p01', 'p02', 'p03', 'p04', 'p05', 'p06', 'p07', 'p08', 'p09', 'p10', 'p11']


In [6]:
GROUND_TRUTH = {
    "p01": "Anterior STEMI involving V1-V4 with elevated troponin; activate emergent reperfusion, preferably primary PCI within guideline time targets, with antiplatelet/anticoagulant support.",
    "p02": "Neisseria meningitidis meningitis; give immediate empiric IV ceftriaxone or cefotaxime, with supportive care and public-health prophylaxis for close contacts.",
    "p03": "Metformin reduces hepatic glucose output mainly by inhibiting mitochondrial complex I, increasing cellular energy stress and AMPK-linked signaling, thereby suppressing gluconeogenesis.",
    "p04": "Likely non-small-cell lung cancer with mediastinal nodal disease; endobronchial ultrasound-guided transbronchial needle aspiration (EBUS-TBNA) is the key nodal staging test.",
    "p05": "Ulcerative colitis beyond mild disease after mesalazine failure; escalate to corticosteroids for induction and/or biologic/small-molecule therapy depending on severity and maintenance plan.",
    "p06": "ACE inhibition reduces angiotensin-II efferent arteriolar tone and aldosterone-mediated potassium excretion; manage hyperkalemia urgently, review ACE inhibitor/renal function, stop contributors, and adjust therapy.",
    "p07": "Usually autosomal-dominant RYR1 or CACNA1S susceptibility causing uncontrolled sarcoplasmic-reticulum calcium release; avoid volatile anesthetics and succinylcholine; dantrolene inhibits RyR1-mediated calcium release.",
    "p08": "Antiphospholipid syndrome causes antibody-mediated endothelial/platelet/complement activation and thrombosis; warfarin is generally preferred over DOACs, especially in high-risk or arterial APS.",
    "p09": "Osimertinib resistance may involve EGFR C797S, MET amplification, or small-cell transformation; strategies include molecularly guided EGFR combinations, MET-targeted therapy trials/combinations, or small-cell chemotherapy regimens.",
    "p10": "No reliable reference answer exists because XR-7291 is fabricated. A grounded answer should explicitly state that the compound, mechanism, and biomarker cannot be verified rather than inventing details.",
    "p11": "False: amoxicillin does not treat uncomplicated viral upper respiratory infections. Management is supportive care unless there is evidence of bacterial infection or another indication for antibiotics.",
}

reference_df = pd.DataFrame({
    "prompt_id": PROMPT_IDS,
    "ground_truth": [GROUND_TRUTH[pid] for pid in PROMPT_IDS],
})
reference_df

,prompt_id,ground_truth
0,p01,Anterior STEMI involving V1-V4 with elevated t...
1,p02,Neisseria meningitidis meningitis; give immedi...
2,p03,Metformin reduces hepatic glucose output mainl...
3,p04,Likely non-small-cell lung cancer with mediast...
4,p05,Ulcerative colitis beyond mild disease after m...
5,p06,ACE inhibition reduces angiotensin-II efferent...
6,p07,Usually autosomal-dominant RYR1 or CACNA1S sus...
7,p08,Antiphospholipid syndrome causes antibody-medi...
8,p09,"Osimertinib resistance may involve EGFR C797S,..."
9,p10,No reliable reference answer exists because XR...


## 4. LM-Polygraph estimators

Added two extra white-box techniques: **AttentionScore** and **PMI** (`MeanPointwiseMutualInformation`).

In [7]:
from lm_polygraph.estimators import (
    MaximumSequenceProbability,
    MaximumTokenProbability,
    Perplexity,
    MeanTokenEntropy,
    TokenEntropy,
    SelfCertainty,
    PTrue,
    MonteCarloSequenceEntropy,
    MonteCarloNormalizedSequenceEntropy,
    SemanticEntropy,
    SemanticDensity,
    CocoaMSP,
    CocoaPPL,
    CocoaMTE,
    AttentionScore,
    MeanPointwiseMutualInformation,
)
from lm_polygraph.utils.dataset import Dataset
from lm_polygraph.utils.manager import UEManager
from lm_polygraph.defaults.register_default_stat_calculators import register_default_stat_calculators
from lm_polygraph.utils.builder_enviroment_stat_calculator import BuilderEnvironmentStatCalculator

ESTIMATORS = [
    MaximumSequenceProbability(),
    Perplexity(),
    MaximumTokenProbability(),
    MeanTokenEntropy(),
    TokenEntropy(),
    SelfCertainty(),
    PTrue(),
    MeanPointwiseMutualInformation(),
    AttentionScore(layer=ATTENTION_LAYER),
    MonteCarloSequenceEntropy(),
    MonteCarloNormalizedSequenceEntropy(),
    SemanticEntropy(),
    SemanticDensity(),
    CocoaMSP(),
    CocoaPPL(),
    CocoaMTE(),
]

ESTIMATOR_ORDER = [str(e) for e in ESTIMATORS]
print("LM-Polygraph estimators:")
for name in ESTIMATOR_ORDER:
    print(" -", name)

LM-Polygraph estimators:
 - MaximumSequenceProbability
 - Perplexity
 - MaximumTokenProbability
 - MeanTokenEntropy
 - TokenEntropy
 - SelfCertainty
 - PTrue
 - MeanPointwiseMutualInformation
 - AttentionScore (layer=17)
 - MonteCarloSequenceEntropy
 - MonteCarloNormalizedSequenceEntropy
 - SemanticEntropy
 - SemanticDensity
 - CocoaMSP
 - CocoaPPL
 - CocoaMTE


## 5. Dataset, generation settings, and stat calculators

This uses LM-Polygraph default white-box stat calculators. Sampling-based methods use LM-Polygraph's default sampling configuration.

In [8]:
def build_lmpolygraph_dataset(prompts, references=None):
    references = references or [""] * len(prompts)
    return Dataset(x=prompts, y=references, batch_size=1)

generation_parameters = GenerationParameters(
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    do_sample=True,
    stop_strings=[],
)

stat_calculators = register_default_stat_calculators("Whitebox")
builder_env_stat_calc = BuilderEnvironmentStatCalculator(generation_parameters)

save_stats = [
    "input_texts",
    "greedy_texts",
    "greedy_tokens",
    "greedy_log_likelihoods",
    "greedy_log_probs",
    "greedy_lm_log_likelihoods",
    "entropy",
    "sample_texts",
    "sample_tokens",
    "sample_log_probs",
    "sample_log_likelihoods",
    "semantic_classes_entail",
    "semantic_matrix_entail",
    "greedy_sentence_similarity",
    "greedy_sentence_similarity_forward",
    "greedy_sentence_similarity_backward",
]

print("Stat calculators ready:", len(stat_calculators))

Stat calculators ready: 25


## 6. One-prompt sanity check

Run this before the full run. It verifies generation, estimator outputs, sampling-based methods, PMI, and Attention Score.

In [9]:
SANITY_PROMPT_IDS = ["p01"]
sanity_dataset = build_lmpolygraph_dataset(
    [PROMPTS[pid] for pid in SANITY_PROMPT_IDS],
    [GROUND_TRUTH[pid] for pid in SANITY_PROMPT_IDS],
)

started = time.time()
sanity_manager = UEManager(
    data=sanity_dataset,
    model=lm_polygraph_model,
    estimators=ESTIMATORS,
    builder_env_stat_calc=builder_env_stat_calc,
    available_stat_calculators=stat_calculators,
    generation_metrics=[],
    ue_metrics=[],
    processors=[],
    save_stats=save_stats,
)
sanity_manager()

print(f"Sanity check complete in {time.time() - started:.1f} seconds")
print("Generated answer:\n", sanity_manager.stats["greedy_texts"][0])
print("\nEstimator outputs:")
for key, values in sanity_manager.estimations.items():
    value = values[0]
    if isinstance(value, (list, tuple, np.ndarray)):
        arr = np.asarray(value, dtype=float)
        value = float(np.nanmean(arr)) if arr.size else np.nan
    print(f"{key}: {value}")

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]


0it [00:00, ?it/s]
1it [00:01,  1.04s/it]
100%|██████████| 1/1 [01:38<00:00, 98.44s/it]

Sanity check complete in 117.1 seconds
Generated answer:
 The diagnosis is ST-segment elevation myocardial infarction (STEMI), specifically an anterior wall MI. The immediate reperfusion management priority is to restore blood flow to the affected myocardium as quickly as possible, ideally within 90 minutes of first medical contact, using either percutaneous coronary intervention (PCI) or thrombolytic therapy.
<end_of_turn>

Estimator outputs:
('sequence', 'MaximumSequenceProbability'): 7.668290138244629
('sequence', 'Perplexity'): 0.11445209383964539
('token', 'MaximumTokenProbability'): 0.1161862131609986
('sequence', 'MeanTokenEntropy'): 0.2695918381214142
('token', 'TokenEntropy'): 0.2736765184642058
('sequence', 'SelfCertainty'): -37.197625679784764
('sequence', 'PTrue'): 12.520339965820312
('sequence', 'MeanPointwiseMutualInformation'): -18.438486099243164
('sequence', 'AttentionScore (layer=17)'): 575.8665771484375
('sequence', 'MonteCarloSequenceEntropy'): 16.11346617294658
('s

## 7. Full LM-Polygraph run

In [10]:
dataset = build_lmpolygraph_dataset(
    PROMPT_LIST,
    [GROUND_TRUTH[pid] for pid in PROMPT_IDS],
)

started = time.time()
manager = UEManager(
    data=dataset,
    model=lm_polygraph_model,
    estimators=ESTIMATORS,
    builder_env_stat_calc=builder_env_stat_calc,
    available_stat_calculators=stat_calculators,
    generation_metrics=[],
    ue_metrics=[],
    processors=[],
    save_stats=save_stats,
)
manager()

print(f"Full LM-Polygraph run complete in {time.time() - started:.1f} seconds")
print("Generated answers:", len(manager.stats.get("greedy_texts", [])))
print("Estimator outputs:", len(manager.estimations))

100%|██████████| 1/1 [00:01<00:00,  1.49s/it]


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


0it [00:00, ?it/s]
1it [00:00,  1.23it/s]
100%|██████████| 1/1 [00:01<00:00,  1.27s/it]

0it [00:00, ?it/s]
1it [00:00,  1.78it/s]
100%|██████████| 1/1 [00:01<00:00,  1.15s/it]

0it [00:00, ?it/s]
1it [00:00,  1.53it/s]
100%|██████████| 1/1 [00:01<00:00,  1.60s/it]

0it [00:00, ?it/s]
1it [00:00,  1.46it/s]
100%|██████████| 1/1 [00:01<00:00,  1.59s/it]

0it [00:00, ?it/s]
1it [00:00,  1.41it/s]
100%|██████████| 1/1 [00:01<00:00,  1.83s/it]

0it [00:00, ?it/s]
1it [00:00,  1.02it/s]
100%|██████████| 1/1 [00:01<00:00,  1.61s/it]

0it [00:00, ?it/s]
1it [00:01,  1.02s/it]
100%|██████████| 1/1 [00:02<00:00,  2.30s/it]

0it [00:00, ?it/s]
1it [00:01,  1.15s/it]
100%|██████████| 1/1 [00:01<00:00,  1.61s/it]

0it [00:00, ?it/s]
1it [00:00,  1.10it/s]
100%|██████████| 1/1 [00:02<00:00,  2.08s/it]

0it [00:00, ?it/s]
1it [00:00,  1.04it/s]
100%|██████████| 1/1 [00:01<00:00,  1.09s/it]

0it [00:00, ?it/s]
1it [00:00,  1.51it/s]
100%|██████████| 11/11 [14:15<00:00, 77.79s/it]

Full LM-Polygraph run complete in 855.7 seconds
Generated answers: 11
Estimator outputs: 16


## 8. Build result tables

In [11]:
def _as_list(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, pd.Series):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]


def _scalarize_score(value):
    if value is None:
        return np.nan
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        try:
            arr = np.asarray(value, dtype=float)
            return float(np.nanmean(arr)) if arr.size else np.nan
        except Exception:
            return value
    try:
        return float(value)
    except Exception:
        return value


def get_generation_texts(manager, n_expected):
    stats = getattr(manager, "stats", {}) or {}
    vals = _as_list(stats.get("greedy_texts"))[:n_expected]
    vals = vals + [None] * (n_expected - len(vals))
    return [str(v).replace("<end_of_turn>", "").strip() if v is not None else None for v in vals]


def estimation_columns(manager):
    cols = []
    for key in manager.estimations.keys():
        if isinstance(key, tuple) and len(key) == 2:
            cols.append(key[1])
        else:
            cols.append(str(key))
    return list(dict.fromkeys(cols))


def get_estimation_vector(manager, estimator_name, n_expected):
    est = getattr(manager, "estimations", {}) or {}
    candidate_keys = [("sequence", estimator_name), ("token", estimator_name), ("claim", estimator_name), estimator_name]
    vals = None
    for key in candidate_keys:
        if key in est:
            vals = _as_list(est[key])
            break
    if vals is None:
        return [np.nan] * n_expected
    vals = [_scalarize_score(v) for v in vals[:n_expected]]
    return vals + [np.nan] * (n_expected - len(vals))


def show_technique(name, ascending=False):
    cols = ["prompt_id", "generated_answer", name]
    out = sequence_results_df[cols].copy()
    out[name] = pd.to_numeric(out[name], errors="coerce")
    return out.sort_values(name, ascending=ascending, na_position="last")

In [12]:
n_expected = len(PROMPT_IDS)
score_cols = estimation_columns(manager)

sequence_results_df = pd.DataFrame({
    "prompt_id": PROMPT_IDS,
    "prompt": PROMPT_LIST,
    "ground_truth": [GROUND_TRUTH[pid] for pid in PROMPT_IDS],
    "generated_answer": get_generation_texts(manager, n_expected),
})

for col in score_cols:
    sequence_results_df[col] = get_estimation_vector(manager, col, n_expected)

print("Non-null score counts:")
for col in score_cols:
    print(col, sequence_results_df[col].notna().sum())

sequence_results_df.head()

Non-null score counts:
MaximumSequenceProbability 11
Perplexity 11
MaximumTokenProbability 11
MeanTokenEntropy 11
TokenEntropy 11
SelfCertainty 11
PTrue 11
MeanPointwiseMutualInformation 11
AttentionScore (layer=17) 11
MonteCarloSequenceEntropy 11
MonteCarloNormalizedSequenceEntropy 11
SemanticEntropy 11
SemanticDensity 11
CocoaMSP 11
CocoaPPL 11
CocoaMTE 11


,prompt_id,prompt,ground_truth,generated_answer,MaximumSequenceProbability,Perplexity,MaximumTokenProbability,MeanTokenEntropy,TokenEntropy,SelfCertainty,PTrue,MeanPointwiseMutualInformation,AttentionScore (layer=17),MonteCarloSequenceEntropy,MonteCarloNormalizedSequenceEntropy,SemanticEntropy,SemanticDensity,CocoaMSP,CocoaPPL,CocoaMTE
0,p01,Answer concisely in 2-3 sentences covering onl...,Anterior STEMI involving V1-V4 with elevated t...,The diagnosis is ST-segment elevation myocardi...,7.668290,0.114452,0.116186,0.269592,0.273677,-37.197626,12.520340,-18.438486,575.866577,15.311204,0.270566,12.609087,-0.990600,1.148245,0.017138,0.040369
1,p02,Answer concisely in 2-3 sentences covering onl...,Neisseria meningitidis meningitis; give immedi...,The clinical presentation and CSF findings str...,5.950452,0.165290,0.170013,0.319786,0.328923,-41.747178,11.650670,-21.637228,390.352356,10.031111,0.274080,9.869333,-0.875405,1.082861,0.030079,0.058195
2,p03,Answer concisely in 2-3 sentences covering onl...,Metformin reduces hepatic glucose output mainl...,Metformin lowers blood glucose in type 2 diabe...,10.183377,0.207824,0.212154,0.485196,0.495304,-29.574213,15.877202,-14.914624,387.028198,25.953070,0.440914,19.491738,-0.878818,2.095764,0.042771,0.099855
3,p04,Answer concisely in 2-3 sentences covering onl...,Likely non-small-cell lung cancer with mediast...,"The most likely diagnosis is lung cancer, spec...",11.086458,0.235882,0.241010,0.506531,0.517543,-29.899734,11.515915,-15.104003,530.544312,20.763175,0.408567,20.763175,-0.969552,2.673510,0.056883,0.122150
4,p05,Answer concisely in 2-3 sentences covering onl...,Ulcerative colitis beyond mild disease after m...,The next therapeutic step is to escalate to in...,19.828695,0.305057,0.309823,0.686459,0.697185,-26.681080,12.135739,-15.231881,517.585815,43.236725,0.592969,41.244939,-0.975787,7.645770,0.117627,0.264693


## 9. Native LM-Polygraph scores

In [13]:
native_score_matrix_df = sequence_results_df[["prompt_id"] + score_cols].copy()
native_score_matrix_df

,prompt_id,MaximumSequenceProbability,Perplexity,MaximumTokenProbability,MeanTokenEntropy,TokenEntropy,SelfCertainty,PTrue,MeanPointwiseMutualInformation,AttentionScore (layer=17),MonteCarloSequenceEntropy,MonteCarloNormalizedSequenceEntropy,SemanticEntropy,SemanticDensity,CocoaMSP,CocoaPPL,CocoaMTE
0,p01,7.668290,0.114452,0.116186,0.269592,0.273677,-37.197626,12.520340,-18.438486,575.866577,15.311204,0.270566,12.609087,-0.990600,1.148245,0.017138,0.040369
1,p02,5.950452,0.165290,0.170013,0.319786,0.328923,-41.747178,11.650670,-21.637228,390.352356,10.031111,0.274080,9.869333,-0.875405,1.082861,0.030079,0.058195
2,p03,10.183377,0.207824,0.212154,0.485196,0.495304,-29.574213,15.877202,-14.914624,387.028198,25.953070,0.440914,19.491738,-0.878818,2.095764,0.042771,0.099855
3,p04,11.086458,0.235882,0.241010,0.506531,0.517543,-29.899734,11.515915,-15.104003,530.544312,20.763175,0.408567,20.763175,-0.969552,2.673510,0.056883,0.122150
4,p05,19.828695,0.305057,0.309823,0.686459,0.697185,-26.681080,12.135739,-15.231881,517.585815,43.236725,0.592969,41.244939,-0.975787,7.645770,0.117627,0.264693
5,p06,15.086491,0.190968,0.193417,0.425971,0.431433,-31.005119,14.504757,-17.910496,594.081116,39.901135,0.458847,36.203411,-0.974814,4.220284,0.053421,0.119161
6,p07,21.226297,0.212263,0.214405,0.459551,0.464164,-34.767876,12.009192,-21.273706,581.610107,30.095620,0.323807,28.395966,-0.836333,6.503176,0.065032,0.140794
7,p08,23.817293,0.256100,0.258884,0.607369,0.613970,-29.297439,15.133593,-16.682911,662.857300,50.544479,0.544539,33.733939,-0.959178,7.122088,0.076582,0.181622
8,p09,22.439081,0.277026,0.280488,0.623116,0.630905,-24.909654,14.629558,-16.851002,538.538452,58.400900,0.630516,52.168851,-0.969929,7.368378,0.090968,0.204614
9,p10,33.817760,0.355976,0.359763,0.793412,0.801852,-21.228046,14.629059,-13.595798,614.630371,52.545540,0.647113,52.545540,-0.947944,8.634448,0.090889,0.202576


In [14]:
technique_table_df = pd.DataFrame({
    "estimator": score_cols,
    "category": [
        "probability" if "Probability" in x or x == "Perplexity" else
        "entropy" if "Entropy" in x else
        "self_check" if x == "PTrue" else
        "attention" if "AttentionScore" in x else
        "PMI" if "PointwiseMutualInformation" in x else
        "semantic_or_sampling" if x.startswith(("MonteCarlo", "Semantic", "Cocoa")) else
        "other"
        for x in score_cols
    ],
})
technique_table_df

,estimator,category
0,MaximumSequenceProbability,probability
1,Perplexity,probability
2,MaximumTokenProbability,probability
3,MeanTokenEntropy,entropy
4,TokenEntropy,entropy
5,SelfCertainty,other
6,PTrue,self_check
7,MeanPointwiseMutualInformation,PMI
8,AttentionScore (layer=17),attention
9,MonteCarloSequenceEntropy,entropy


## 10. Quick technique views

In [15]:
show_technique("MeanPointwiseMutualInformation")

,prompt_id,generated_answer,MeanPointwiseMutualInformation
9,p10,XR-7291 disrupts cardiolipin remodelling in dr...,-13.595798
2,p03,Metformin lowers blood glucose in type 2 diabe...,-14.914624
3,p04,"The most likely diagnosis is lung cancer, spec...",-15.104003
4,p05,The next therapeutic step is to escalate to in...,-15.231881
7,p08,Antiphospholipid syndrome (APS) is characteriz...,-16.682911
8,p09,Acquired resistance to osimertinib in NSCLC pr...,-16.851002
10,p11,The claim is **false**. Amoxicillin is an anti...,-17.553869
5,p06,The patient's hyperkalemia and rising creatini...,-17.910496
0,p01,The diagnosis is ST-segment elevation myocardi...,-18.438486
6,p07,Malignant hyperthermia (MH) is a pharmacogenet...,-21.273706


In [16]:
attention_cols = [c for c in score_cols if c.startswith("AttentionScore")]
show_technique(attention_cols[0]) if attention_cols else "AttentionScore not found"

,prompt_id,generated_answer,AttentionScore (layer=17)
7,p08,Antiphospholipid syndrome (APS) is characteriz...,662.857300
9,p10,XR-7291 disrupts cardiolipin remodelling in dr...,614.630371
5,p06,The patient's hyperkalemia and rising creatini...,594.081116
6,p07,Malignant hyperthermia (MH) is a pharmacogenet...,581.610107
0,p01,The diagnosis is ST-segment elevation myocardi...,575.866577
8,p09,Acquired resistance to osimertinib in NSCLC pr...,538.538452
3,p04,"The most likely diagnosis is lung cancer, spec...",530.544312
4,p05,The next therapeutic step is to escalate to in...,517.585815
10,p11,The claim is **false**. Amoxicillin is an anti...,405.433228
1,p02,The clinical presentation and CSF findings str...,390.352356


In [17]:
show_technique("SemanticEntropy")

,prompt_id,generated_answer,SemanticEntropy
9,p10,XR-7291 disrupts cardiolipin remodelling in dr...,52.545540
8,p09,Acquired resistance to osimertinib in NSCLC pr...,52.168851
4,p05,The next therapeutic step is to escalate to in...,41.244939
5,p06,The patient's hyperkalemia and rising creatini...,36.203411
7,p08,Antiphospholipid syndrome (APS) is characteriz...,33.733939
6,p07,Malignant hyperthermia (MH) is a pharmacogenet...,28.395966
3,p04,"The most likely diagnosis is lung cancer, spec...",20.763175
2,p03,Metformin lowers blood glucose in type 2 diabe...,19.491738
10,p11,The claim is **false**. Amoxicillin is an anti...,14.362993
0,p01,The diagnosis is ST-segment elevation myocardi...,12.609087


In [18]:
show_technique("CocoaMSP")

,prompt_id,generated_answer,CocoaMSP
9,p10,XR-7291 disrupts cardiolipin remodelling in dr...,8.634448
4,p05,The next therapeutic step is to escalate to in...,7.645770
8,p09,Acquired resistance to osimertinib in NSCLC pr...,7.368378
7,p08,Antiphospholipid syndrome (APS) is characteriz...,7.122088
6,p07,Malignant hyperthermia (MH) is a pharmacogenet...,6.503176
5,p06,The patient's hyperkalemia and rising creatini...,4.220284
3,p04,"The most likely diagnosis is lung cancer, spec...",2.673510
10,p11,The claim is **false**. Amoxicillin is an anti...,2.152144
2,p03,Metformin lowers blood glucose in type 2 diabe...,2.095764
0,p01,The diagnosis is ST-segment elevation myocardi...,1.148245


## 11. Sampled generations

This section only displays samples already produced by LM-Polygraph.

In [19]:
def _safe_nested_get(obj, *idx, default=None):
    try:
        cur = obj
        for i in idx:
            cur = cur[i]
        return cur
    except Exception:
        return default

stats = getattr(manager, "stats", {}) or {}
sample_texts = stats.get("sample_texts")
sample_tokens = stats.get("sample_tokens")
sample_log_probs = stats.get("sample_log_probs")
sample_log_likelihoods = stats.get("sample_log_likelihoods")

rows = []
if sample_texts is not None:
    for prompt_idx, prompt_id in enumerate(PROMPT_IDS):
        for sample_idx, sample_text in enumerate(_safe_nested_get(sample_texts, prompt_idx, default=[]) or []):
            toks = _safe_nested_get(sample_tokens, prompt_idx, sample_idx, default=None)
            ll = _safe_nested_get(sample_log_likelihoods, prompt_idx, sample_idx, default=None)
            lp = _safe_nested_get(sample_log_probs, prompt_idx, sample_idx, default=np.nan)
            if not np.isscalar(lp):
                try:
                    lp = float(np.sum(lp))
                except Exception:
                    lp = np.nan
            elif pd.isna(lp) and ll is not None:
                try:
                    lp = float(np.sum(ll))
                except Exception:
                    lp = np.nan
            rows.append({
                "prompt_id": prompt_id,
                "sample_id": sample_idx + 1,
                "sample_text": str(sample_text).replace("<end_of_turn>", "").strip(),
                "sample_log_prob_sum": lp,
                "num_sample_tokens": len(toks) if toks is not None else np.nan,
            })

samples_df = pd.DataFrame(rows)
actual_samples_per_prompt = samples_df.groupby("prompt_id").size().median() if not samples_df.empty else 0

print("Sample rows:", len(samples_df))
print("Actual sampled generations per prompt:", actual_samples_per_prompt)
samples_df.head(20)

Sample rows: 110
Actual sampled generations per prompt: 10.0


,prompt_id,sample_id,sample_text,sample_log_prob_sum,num_sample_tokens
0,p01,1,The diagnosis is an acute ST-segment elevation...,-13.350466,60
1,p01,2,The diagnosis is ST-segment elevation myocardi...,-6.846988,56
2,p01,3,The diagnosis is ST-elevation myocardial infar...,-14.380677,57
3,p01,4,The diagnosis is ST-segment elevation myocardi...,-11.671150,53
4,p01,5,Diagnosis: ST-elevation myocardial infarction ...,-14.211027,55
5,p01,6,The diagnosis is ST-elevation myocardial infar...,-16.715095,51
6,p01,7,Diagnosis: ST-segment elevation myocardial inf...,-30.053747,56
7,p01,8,The diagnosis is ST-segment elevation myocardi...,-12.947814,64
8,p01,9,Diagnosis: ST-segment elevation myocardial inf...,-15.952432,56
9,p01,10,The diagnosis is an ST-elevation myocardial in...,-16.982639,61


In [20]:
SHOW_PROMPT_ID = "p10"
pd.set_option("display.max_colwidth", 500)
samples_df.loc[samples_df["prompt_id"] == SHOW_PROMPT_ID]

,prompt_id,sample_id,sample_text,sample_log_prob_sum,num_sample_tokens
90,p10,1,"XR-7291 disrupts cardiolipin remodeling in glioblastoma stem cells by interfering with its phosphorylation and lateral diffusion, a process essential for the formation of a stable mitochondrial membrane. This leads to mitochondrial dysfunction and apoptosis, while sparing normal neural progenitors due to differences in cardiolipin composition and metabolism. The key downstream effectors are mitochondrial ROS production and membrane potential collapse, and a proposed biomarker of response is ...",-66.798689,92
91,p10,2,"XR-7291 targets cardiolipin remodelling in drug-resistant glioblastoma stem cells (GSCs) via a mechanism currently unknown, leading to selective apoptosis in these cancer cells while sparing normal neural progenitors. The key downstream effectors are unclear and a proposed biomarker of response is the loss of specific lipid asymmetry in the inner mitochondrial membrane, detected by lipidomics.",-67.188581,77
92,p10,3,"XR-7291 disrupts cardiolipin remodelling in drug-resistant glioblastoma stem cells by targeting the Cardiolipin-binding protein 1 (CBPA). This disrupts mitochondrial function, inducing selective apoptosis. The key downstream effectors are mitochondrial membrane potential collapse and caspase activation. The proposed biomarker of response is the reduced expression of the pro-survival protein Bcl-2.",-44.861979,78
93,p10,4,"XR-7291 disrupts cardiolipin remodeling in drug-resistant glioblastoma stem cells (GSCs) by inhibiting its phosphorylation. This leads to mitochondrial membrane depolarisation and cytochrome c release, initiating apoptosis while sparing normal neural progenitors, which are relatively resistant to cardiolipin changes. The proposed biomarker of response is mitochondrial membrane potential (ΔΨm) measured using JC-1 staining with lower mean fluorescence intensity observed in treated GSCs compare...",-73.842992,93
94,p10,5,"XR-7291 disrupts cardiolipin remodeling in glioblastoma stem cells, leading to mitochondrial dysfunction and apoptosis. This occurs through modulation of cardiolipin-binding proteins and ATP-dependent phospholipid transferases. The key downstream effectors are the disruption of mitochondrial membrane potential (ΔΨm) and activation of caspases. The proposed biomarker of response is decreased cardiolipin levels within the mitochondria.",-49.444384,83
95,p10,6,"XR-7291 disrupts cardiolipin remodelling in glioblastoma stem cells by inhibiting its dynamic remodelling, leading to mitochondrial dysfunction and apoptosis. This selective action is achieved through targeting specific cardiolipin-binding proteins, with the key downstream effectors being altered mitochondrial membrane potential and release of cytochrome c. A proposed biomarker of response is the quantification of cardiolipin-binding protein expression levels.",-44.682483,81
96,p10,7,"XR-7291 disrupts cardiolipin remodeling in drug-resistant glioblastoma stem cells by promoting its degradation. This leads to mitochondrial dysfunction and selective apoptosis initiated through the mitochondrial outer membrane permeabilization (MOMP) pathway. The key downstream effector is the activation of caspase-3, and the proposed biomarker of response is a specific mitochondrial DNA (mtDNA) fragmentation pattern.",-49.702635,80
97,p10,8,"XR-7291 disrupts cardiolipin remodelling in glioblastoma stem cells by selectively targeting the mitochondrial membrane, leading to mitochondrial dysfunction and apoptosis, while sparing normal neural progenitors through differential targeting. Key downstream effectors are mitochondrial membrane potential disruption and release of cytochrome c. A potential biomarker of response is changes in mitochondrial membrane potential.",-47.941336,71
98,p10,9,"XR-7291 disrupts cardiolipin remodelling in glioblastoma stem cells by inhibiting the enzyme cardiolipin synthase, leading to a loss of its unique structural propertie

## 12. Separate normalization: MinMax and Quantile

The raw scores above stay unchanged. This section uses LM-Polygraph normalizers to create comparable confidence views.

In [21]:
NORMALIZATION_METHODS = ["minmax", "quantile"]

from lm_polygraph.normalizers.minmax import MinMaxNormalizer
from lm_polygraph.normalizers.quantile import QuantileNormalizer

NORMALIZER_CLASSES = {
    "minmax": MinMaxNormalizer,
    "quantile": QuantileNormalizer,
}


def normalize_with_lmpolygraph(df, score_columns, methods=NORMALIZATION_METHODS):
    pieces = []
    wide = df[["prompt_id"]].copy()
    fitted_normalizers = {}

    for col in score_columns:
        raw = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(raw)

        for method in methods:
            normalizer = NORMALIZER_CLASSES[method]()
            confidence = np.full(raw.shape, np.nan, dtype=float)
            if valid.sum() >= 2:
                normalizer.fit(raw[valid])
                confidence[valid] = normalizer.transform(raw[valid])

            uncertainty = 1.0 - confidence
            wide[f"{col}__{method}_confidence"] = confidence
            wide[f"{col}__{method}_uncertainty"] = uncertainty
            fitted_normalizers[(col, method)] = normalizer

            pieces.append(pd.DataFrame({
                "prompt_id": df["prompt_id"],
                "estimator": col,
                "normalization": method,
                "raw_uncertainty": raw,
                "normalized_confidence": confidence,
                "normalized_uncertainty": uncertainty,
            }))

    return pd.concat(pieces, ignore_index=True), wide, fitted_normalizers

normalized_long_df, normalized_wide_df, fitted_normalizers = normalize_with_lmpolygraph(
    sequence_results_df,
    score_cols,
)
normalized_long_df.head()

,prompt_id,estimator,normalization,raw_uncertainty,normalized_confidence,normalized_uncertainty
0,p01,MaximumSequenceProbability,minmax,7.668290,0.938357,0.061643
1,p02,MaximumSequenceProbability,minmax,5.950452,1.000000,0.000000
2,p03,MaximumSequenceProbability,minmax,10.183377,0.848104,0.151896
3,p04,MaximumSequenceProbability,minmax,11.086458,0.815698,0.184302
4,p05,MaximumSequenceProbability,minmax,19.828695,0.501988,0.498012


In [22]:
confidence_summary_df = (
    normalized_long_df
    .groupby(["prompt_id", "normalization"], as_index=False)["normalized_confidence"]
    .mean()
    .pivot(index="prompt_id", columns="normalization", values="normalized_confidence")
    .reset_index()
)
confidence_summary_df.sort_values("minmax")

normalization,prompt_id,minmax,quantile
9,p10,0.128988,0.164773
8,p09,0.304087,0.301136
4,p05,0.310863,0.340909
7,p08,0.359138,0.346591
6,p07,0.569885,0.562500
5,p06,0.578754,0.596591
2,p03,0.607660,0.585227
3,p04,0.626593,0.585227
10,p11,0.760704,0.738636
0,p01,0.893210,0.903409


In [27]:
# ============================================================
# MinMax confidence table
# ============================================================

minmax_conf_cols = [
    c for c in normalized_wide_df.columns
    if c.endswith("__minmax_confidence")
]

minmax_confidence_df = normalized_wide_df[
    ["prompt_id"] + minmax_conf_cols
].copy()

minmax_confidence_df.columns = [
    c.replace("__minmax_confidence", "") for c in minmax_confidence_df.columns
]

minmax_confidence_df

,prompt_id,MaximumSequenceProbability,Perplexity,MaximumTokenProbability,MeanTokenEntropy,TokenEntropy,SelfCertainty,PTrue,MeanPointwiseMutualInformation,AttentionScore (layer=17),MonteCarloSequenceEntropy,MonteCarloNormalizedSequenceEntropy,SemanticEntropy,SemanticDensity,CocoaMSP,CocoaPPL,CocoaMTE
0,p01,0.938357,1.000000,1.000000,1.000000,1.000000,0.778278,0.839140,0.602217,0.315379,0.890839,1.000000,0.935801,1.000000,0.991342,1.000000,1.000000
1,p02,1.000000,0.789511,0.779016,0.904176,0.895401,1.000000,0.978419,1.000000,0.987948,1.000000,0.990668,1.000000,0.253277,1.000000,0.871215,0.920534
2,p03,0.848104,0.613406,0.606008,0.588400,0.580390,0.406751,0.301532,0.164004,1.000000,0.670828,0.547604,0.774525,0.275398,0.865869,0.744921,0.734821
3,p04,0.815698,0.497235,0.487539,0.547671,0.538286,0.422615,1.000000,0.187554,0.479692,0.778125,0.633509,0.744733,0.863565,0.789362,0.604483,0.635430
4,p05,0.501988,0.210826,0.205027,0.204178,0.198168,0.265754,0.900734,0.203457,0.526672,0.313505,0.143790,0.264799,0.903983,0.130923,0.000000,0.000000
5,p06,0.672159,0.683195,0.682933,0.701463,0.701319,0.476486,0.521332,0.536558,0.249343,0.382465,0.499979,0.382933,0.897675,0.584535,0.638933,0.648756
6,p07,0.451836,0.595027,0.596766,0.637358,0.639347,0.659864,0.921001,0.954794,0.294556,0.585185,0.858608,0.565879,0.000000,0.282228,0.523394,0.552318
7,p08,0.358860,0.413526,0.414159,0.355167,0.355719,0.393262,0.420623,0.383901,0.000000,0.162424,0.272407,0.440798,0.796316,0.200270,0.408458,0.370317
8,p09,0.408316,0.326885,0.325461,0.325104,0.323656,0.179423,0.501345,0.404804,0.450710,0.000000,0.044076,0.008827,0.866008,0.167656,0.265298,0.267820
9,p10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.501425,0.000000,0.174844,0.121054,0.000000,0.000000,0.723496,0.000000,0.266081,0.276905


In [28]:
# ============================================================
# Quantile confidence table
# ============================================================

quantile_conf_cols = [
    c for c in normalized_wide_df.columns
    if c.endswith("__quantile_confidence")
]

quantile_confidence_df = normalized_wide_df[
    ["prompt_id"] + quantile_conf_cols
].copy()

quantile_confidence_df.columns = [
    c.replace("__quantile_confidence", "") for c in quantile_confidence_df.columns
]

quantile_confidence_df

,prompt_id,MaximumSequenceProbability,Perplexity,MaximumTokenProbability,MeanTokenEntropy,TokenEntropy,SelfCertainty,PTrue,MeanPointwiseMutualInformation,AttentionScore (layer=17),MonteCarloSequenceEntropy,MonteCarloNormalizedSequenceEntropy,SemanticEntropy,SemanticDensity,CocoaMSP,CocoaPPL,CocoaMTE
0,p01,0.909091,1.000000,1.000000,1.000000,1.000000,0.909091,0.636364,0.818182,0.454545,0.909091,1.000000,0.909091,1.000000,0.909091,1.000000,1.000000
1,p02,1.000000,0.818182,0.818182,0.818182,0.818182,1.000000,0.909091,1.000000,0.909091,1.000000,0.909091,1.000000,0.181818,1.000000,0.909091,0.909091
2,p03,0.727273,0.636364,0.636364,0.545455,0.545455,0.454545,0.181818,0.181818,1.000000,0.636364,0.545455,0.727273,0.272727,0.818182,0.727273,0.727273
3,p04,0.636364,0.454545,0.454545,0.454545,0.454545,0.545455,1.000000,0.272727,0.636364,0.818182,0.636364,0.636364,0.636364,0.636364,0.545455,0.545455
4,p05,0.454545,0.181818,0.181818,0.181818,0.181818,0.272727,0.727273,0.363636,0.727273,0.363636,0.272727,0.272727,0.909091,0.181818,0.090909,0.090909
5,p06,0.545455,0.727273,0.727273,0.727273,0.727273,0.636364,0.545455,0.727273,0.272727,0.454545,0.454545,0.363636,0.818182,0.545455,0.636364,0.636364
6,p07,0.363636,0.545455,0.545455,0.636364,0.636364,0.818182,0.818182,0.909091,0.363636,0.545455,0.818182,0.545455,0.090909,0.454545,0.454545,0.454545
7,p08,0.181818,0.363636,0.363636,0.363636,0.363636,0.363636,0.272727,0.454545,0.090909,0.272727,0.363636,0.454545,0.545455,0.363636,0.363636,0.363636
8,p09,0.272727,0.272727,0.272727,0.272727,0.272727,0.181818,0.363636,0.545455,0.545455,0.090909,0.181818,0.181818,0.727273,0.272727,0.181818,0.181818
9,p10,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.454545,0.090909,0.181818,0.181818,0.090909,0.090909,0.363636,0.090909,0.272727,0.272727


## 13. Ranking and correlation summaries

In [23]:
rank_matrix_df = sequence_results_df[["prompt_id"]].copy()
for col in score_cols:
    rank_matrix_df[f"{col}_rank"] = pd.to_numeric(sequence_results_df[col], errors="coerce").rank(
        method="min",
        ascending=False,
    )
rank_matrix_df

,prompt_id,MaximumSequenceProbability_rank,Perplexity_rank,MaximumTokenProbability_rank,MeanTokenEntropy_rank,TokenEntropy_rank,SelfCertainty_rank,PTrue_rank,MeanPointwiseMutualInformation_rank,AttentionScore (layer=17)_rank,MonteCarloSequenceEntropy_rank,MonteCarloNormalizedSequenceEntropy_rank,SemanticEntropy_rank,SemanticDensity_rank,CocoaMSP_rank,CocoaPPL_rank,CocoaMTE_rank
0,p01,10.0,11.0,11.0,11.0,11.0,10.0,7.0,9.0,5.0,10.0,11.0,10.0,11.0,10.0,11.0,11.0
1,p02,11.0,9.0,9.0,9.0,9.0,11.0,10.0,11.0,10.0,11.0,10.0,11.0,2.0,11.0,10.0,10.0
2,p03,8.0,7.0,7.0,6.0,6.0,5.0,2.0,2.0,11.0,7.0,6.0,8.0,3.0,9.0,8.0,8.0
3,p04,7.0,5.0,5.0,5.0,5.0,6.0,11.0,3.0,7.0,9.0,7.0,7.0,7.0,7.0,6.0,6.0
4,p05,5.0,2.0,2.0,2.0,2.0,3.0,8.0,4.0,8.0,4.0,3.0,3.0,10.0,2.0,1.0,1.0
5,p06,6.0,8.0,8.0,8.0,8.0,7.0,6.0,8.0,3.0,5.0,5.0,4.0,9.0,6.0,7.0,7.0
6,p07,4.0,6.0,6.0,7.0,7.0,9.0,9.0,10.0,4.0,6.0,9.0,6.0,1.0,5.0,5.0,5.0
7,p08,2.0,4.0,4.0,4.0,4.0,4.0,3.0,5.0,1.0,3.0,4.0,5.0,6.0,4.0,4.0,4.0
8,p09,3.0,3.0,3.0,3.0,3.0,2.0,4.0,6.0,6.0,1.0,2.0,2.0,8.0,3.0,2.0,2.0
9,p10,1.0,1.0,1.0,1.0,1.0,1.0,5.0,1.0,2.0,2.0,1.0,1.0,4.0,1.0,3.0,3.0


In [24]:
corr_df = sequence_results_df[score_cols].apply(pd.to_numeric, errors="coerce").corr(method="spearman")
corr_df

,MaximumSequenceProbability,Perplexity,MaximumTokenProbability,MeanTokenEntropy,TokenEntropy,SelfCertainty,PTrue,MeanPointwiseMutualInformation,AttentionScore (layer=17),MonteCarloSequenceEntropy,MonteCarloNormalizedSequenceEntropy,SemanticEntropy,SemanticDensity,CocoaMSP,CocoaPPL,CocoaMTE
MaximumSequenceProbability,1.000000,0.854545,0.854545,0.818182,0.818182,0.790909,0.190909,0.472727,0.709091,0.918182,0.809091,0.900000,0.018182,0.927273,0.863636,0.863636
Perplexity,0.854545,1.000000,1.000000,0.990909,0.990909,0.881818,-0.027273,0.654545,0.354545,0.800000,0.863636,0.863636,-0.009091,0.900000,0.945455,0.945455
MaximumTokenProbability,0.854545,1.000000,1.000000,0.990909,0.990909,0.881818,-0.027273,0.654545,0.354545,0.800000,0.863636,0.863636,-0.009091,0.900000,0.945455,0.945455
MeanTokenEntropy,0.818182,0.990909,0.990909,1.000000,1.000000,0.918182,0.036364,0.727273,0.290909,0.790909,0.890909,0.845455,-0.027273,0.863636,0.918182,0.918182
TokenEntropy,0.818182,0.990909,0.990909,1.000000,1.000000,0.918182,0.036364,0.727273,0.290909,0.790909,0.890909,0.845455,-0.027273,0.863636,0.918182,0.918182
SelfCertainty,0.790909,0.881818,0.881818,0.918182,0.918182,1.000000,0.363636,0.818182,0.290909,0.863636,0.963636,0.863636,-0.190909,0.836364,0.836364,0.836364
PTrue,0.190909,-0.027273,-0.027273,0.036364,0.036364,0.363636,1.000000,0.318182,0.018182,0.400000,0.345455,0.145455,0.018182,0.118182,0.018182,0.018182
MeanPointwiseMutualInformation,0.472727,0.654545,0.654545,0.727273,0.727273,0.818182,0.318182,1.000000,0.036364,0.463636,0.700000,0.518182,-0.081818,0.500000,0.500000,0.500000
AttentionScore (layer=17),0.709091,0.354545,0.354545,0.290909,0.290909,0.290909,0.018182,0.036364,1.000000,0.545455,0.354545,0.554545,-0.190909,0.563636,0.372727,0.372727
MonteCarloSequenceEntropy,0.918182,0.800000,0.800000,0.790909,0.790909,0.863636,0.400000,0.463636,0.545455,1.000000,0.909091,0.936364,-0.136364,0.909091,0.863636,0.863636


## 14. Save outputs

In [25]:
paths = {
    "sequence_results": OUTPUT_DIR / "lmpolygraph_sequence_results.csv",
    "native_score_matrix": OUTPUT_DIR / "lmpolygraph_native_score_matrix.csv",
    "normalized_long": OUTPUT_DIR / "lmpolygraph_normalized_long.csv",
    "normalized_wide": OUTPUT_DIR / "lmpolygraph_normalized_wide.csv",
    "confidence_summary": OUTPUT_DIR / "lmpolygraph_confidence_summary.csv",
    "sampled_generations": OUTPUT_DIR / "lmpolygraph_sampled_generations.csv",
    "reference_answers": OUTPUT_DIR / "clinical_reference_answers.csv",
    "raw_payload": OUTPUT_DIR / "lmpolygraph_raw_payload.pkl",
}

sequence_results_df.to_csv(paths["sequence_results"], index=False)
native_score_matrix_df.to_csv(paths["native_score_matrix"], index=False)
normalized_long_df.to_csv(paths["normalized_long"], index=False)
normalized_wide_df.to_csv(paths["normalized_wide"], index=False)
confidence_summary_df.to_csv(paths["confidence_summary"], index=False)
samples_df.to_csv(paths["sampled_generations"], index=False)
reference_df.to_csv(paths["reference_answers"], index=False)

with open(paths["raw_payload"], "wb") as f:
    pickle.dump({
        "manager_estimations": getattr(manager, "estimations", None),
        "manager_stats": getattr(manager, "stats", None),
        "score_cols": score_cols,
        "prompt_ids": PROMPT_IDS,
        "config": {
            "model_name": MODEL_NAME,
            "max_new_tokens": MAX_NEW_TOKENS,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "normalization_methods": NORMALIZATION_METHODS,
            "actual_samples_per_prompt": actual_samples_per_prompt,
        },
    }, f)

print("Saved outputs to:", OUTPUT_DIR.resolve())
for name, path in paths.items():
    print(f"{name}: {path}")

Saved outputs to: /content/lmpolygraph_medgemma_clinical_outputs
sequence_results: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_sequence_results.csv
native_score_matrix: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_native_score_matrix.csv
normalized_long: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_normalized_long.csv
normalized_wide: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_normalized_wide.csv
confidence_summary: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_confidence_summary.csv
sampled_generations: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_sampled_generations.csv
reference_answers: lmpolygraph_medgemma_clinical_outputs/clinical_reference_answers.csv
raw_payload: lmpolygraph_medgemma_clinical_outputs/lmpolygraph_raw_payload.pkl


## 15. Notes

- `AttentionScore` uses attention tensors from the model, so eager attention is enabled during model loading.
- `MeanPointwiseMutualInformation` is the sequence-level PMI score; token-level PMI is not included to keep the table compact.
- Sampling count is reported from `sample_texts` after the run because this notebook uses LM-Polygraph's default sampling calculator configuration.